# 04 — Generation Demo

Load a trained checkpoint and generate text from custom prompts.
Compare outputs across different temperatures and top-k values.

In [1]:
import sys
sys.path.insert(0, '../..')

import torch
import tiktoken

from src.infra.config import load_config
from src.infra.io import load_checkpoint
from src.models.gpt.model import GPT
from src.core.generation import generate

In [2]:
# ── Load config and checkpoint ────────────────────────────────────────────
config = load_config('../../configs/experiments/exp_001_baseline.yaml')
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = GPT(config.model).to(device)
ckpt_path = '../../outputs/miniGPT/checkpoints/best_model.pt'
load_checkpoint(ckpt_path, model, device=device)
model.eval()

enc = tiktoken.get_encoding('gpt2')
print(f'Model loaded on {device}')

Model loaded on cpu


In [3]:
# ── Generate text ─────────────────────────────────────────────────────────
def generate_text(prompt: str, max_tokens: int = 200,
                  temperature: float = 1.0, top_k: int = 50) -> str:
    tokens = enc.encode_ordinary(prompt)
    ctx = torch.tensor(tokens, dtype=torch.long, device=device).unsqueeze(0)
    out = generate(model, ctx, max_new_tokens=max_tokens,
                   temperature=temperature, top_k=top_k)
    return enc.decode(out.squeeze().tolist())

prompts = [
    'Once upon a time there was a pumpkin.',
    'A little girl went to the woods',
]
for p in prompts:
    print(f'--- Prompt: "{p}" ---')
    print(generate_text(p, max_tokens=200))
    print()

--- Prompt: "Once upon a time there was a pumpkin." ---
Once upon a time there was a pumpkin. The jellyfish was very sad. One day, the tomato decided to go a walk. The ant was sad but could not find it. He stayed on the ground and found some other birds.

The pumpkin made the chicken be hurt, and the egg was very sad. They didn't know how to be. Then, the jellyfish saw a way to get back outside, and he started to feel better.

The egg had stopped and the sun stopped. The jellyfish had come back and it was very comfortable. The chicken started to feel the sun again, and it was time to go home for bed.

At the end of the day, the ant was happy. He felt happy that he had saved the little bug when he finished being up.Once upon a time, there was a little boy named Timmy. He loved her very much and would often take her to find a new toy. One day, Timmy's mom asked him to help her find the

--- Prompt: "A little girl went to the woods" ---
A little girl went to the woods for a walk. When she

In [4]:
# ── Temperature sweep ─────────────────────────────────────────────────────
prompt = 'Once upon a time'
for temp in [0.5, 0.8, 1.0, 1.2]:
    print(f'--- temperature={temp} ---')
    print(generate_text(prompt, max_tokens=80, temperature=temp, top_k=50))
    print()

--- temperature=0.5 ---
Once upon a time, there was a little girl named Lily. She loved to play with her toys all day long. One day, she saw a big tree with lots of leaves. She asked her mom, "Mom, can I have some leaves?"

Her mom laughed and said, "No, Lily. You have to be careful to take a closer look."

Lily thought for a moment and

--- temperature=0.8 ---
Once upon a time, there was a big bird. But a little bird was very small and had a very long wing. One day, the bird had a big wing and was very scared. 

Bob felt sad because he lost his wing. He had lost his wing and couldn't help. 

The bird flew out and saw a branch. The bird was very sad. Bob wanted to help the

--- temperature=1.0 ---
Once upon a time, there was a boy named Jim. He was three years old and loved to play in his garden. He put on his special yellow hat and wore it all day outside him. One day, he found a big tree filled with all of his wings.

Bob was so happy and knew it would never get to work right away. H